# Alpamayo 2 Super Meta-Action Inference

Run this notebook from the repository root with the `Alpamayo 2 Super` kernel. Set `ALPAMAYO2_SUPER_MODEL_ID` to a Hugging Face model id or local release checkpoint before starting the kernel. The notebook selects the validated six-camera/four-frame meta-action input profile before model preparation.

In [ ]:
import os

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from IPython.display import display

from alpamayo2_super import helper
from alpamayo2_super.common.constants import PUBLIC_MODEL_ID
from alpamayo2_super.inference_smoke import resolve_project_path, validate_model_id
from alpamayo2_super.input_profiles import select_task_input
from alpamayo2_super.load_physical_aiavdataset import load_physical_aiavdataset
from alpamayo2_super.models.alpamayo2_super import Alpamayo2Super
from alpamayo2_super.text_tasks import generate_text, prepare_text_generation_inputs
from alpamayo2_super.visualization import plot_meta_action_result

In [ ]:
cwd = Path.cwd()
if (cwd / "examples").exists():
    project_root = cwd
elif (cwd.parent / "examples").exists():
    project_root = cwd.parent
else:
    project_root = cwd

MODEL_ID = os.environ.get("ALPAMAYO2_SUPER_MODEL_ID", PUBLIC_MODEL_ID)
MANIFEST = resolve_project_path(
    os.environ.get("ALPAMAYO2_SUPER_VALIDATION_MANIFEST", "examples/validation_samples.json"),
    project_root,
)
SAMPLE_INDEX = int(os.environ.get("ALPAMAYO2_SUPER_SAMPLE_INDEX", "0"))
MAX_NEW_TOKENS = int(os.environ.get("ALPAMAYO2_SUPER_META_ACTION_MAX_NEW_TOKENS", "512"))
SEED = int(os.environ.get("ALPAMAYO2_SUPER_SEED", "42"))
OUTPUT_DIR = resolve_project_path(
    os.environ.get("ALPAMAYO2_SUPER_OUTPUT_DIR", "outputs"), project_root
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sample = json.loads(MANIFEST.read_text(encoding="utf-8"))["samples"][SAMPLE_INDEX]
clip_id = os.environ.get("ALPAMAYO2_SUPER_CLIP_ID", sample["clip_id"])
t0_us = int(os.environ.get("ALPAMAYO2_SUPER_T0_US", str(sample["t0_us"])))
validate_model_id(MODEL_ID)
if not torch.cuda.is_available():
    raise RuntimeError("Alpamayo 2 Super meta-action inference requires a CUDA GPU.")

print("model:", MODEL_ID)
print("sample:", SAMPLE_INDEX, clip_id, t0_us)

In [ ]:
source_data = load_physical_aiavdataset(
    clip_id,
    t0_us=t0_us,
)
data = select_task_input(source_data, "meta_action")
print("camera_indices:", data["camera_indices"].tolist())

In [ ]:
model = Alpamayo2Super.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="cuda:0")
task_inputs = prepare_text_generation_inputs(
    data=data,
    model_config=model.config,
    tokenizer=model.tokenizer,
    task="meta_action",
)
task_inputs = helper.to_device(task_inputs, "cuda")

In [ ]:
torch.cuda.manual_seed_all(SEED)
with torch.autocast("cuda", dtype=torch.bfloat16):
    result = generate_text(
        model,
        task_inputs,
        top_p=0.98,
        temperature=0.6,
        max_new_tokens=MAX_NEW_TOKENS,
    )

cot = result["cot"][0]
meta_action = result["meta_action"][0]
print("Chain-of-Causation:\n", cot)
print("\nMeta-action:\n", meta_action)

In [ ]:
artifact_stem = f"meta_action_sample{SAMPLE_INDEX}_{clip_id}_{t0_us}"
figure_path = OUTPUT_DIR / f"{artifact_stem}.png"
json_path = OUTPUT_DIR / f"{artifact_stem}.json"
fig, figure_metadata = plot_meta_action_result(
    data=data,
    cot=cot,
    meta_action=meta_action,
    output_path=figure_path,
    model_id=MODEL_ID,
    seed=SEED,
)
display(fig)
plt.close(fig)
payload = {
    "task": "meta_action",
    "model_id": MODEL_ID,
    "clip_id": clip_id,
    "t0_us": t0_us,
    "seed": SEED,
    "max_new_tokens": MAX_NEW_TOKENS,
    "cot": cot,
    "meta_action": meta_action,
    "raw_output": result["raw_outputs"][0],
    "figure_path": str(figure_path),
    "figure_metadata": figure_metadata,
}
json_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("saved_png:", figure_path)
print("saved_json:", json_path)